# 06. Experiment E: cross-encoder rerank

Part of Step 6's seven-notebook series: [01](01_dataset_and_chunking.ipynb) dataset/chunking, [02](02_experiment_a_chunking_strategy.ipynb) Experiment A, [03](03_experiment_b_search_mode.ipynb) Experiment B, [04](04_experiment_c_embedding_model.ipynb) Experiment C, [05](05_experiment_d_weights_and_rrf.ipynb) Experiment D, [06](06_experiment_e_rerank.ipynb) Experiment E, [07](07_calibration_and_analysis.ipynb) calibration/PCA/latency/failure analysis/conclusions. Each runs standalone against its own fresh ingest (via `_setup.py`); none depends on another's kernel state.

`cross-encoder/ms-marco-MiniLM-L6-v2` scoring every candidate in the filtered
pool (Step 11's R19: no more top-20 subset), replacing the calibrated cosine
similarity with the cross-encoder's raw logit mapped through the same
`calibrate()` and fixed per-model bounds cosine similarity uses
(`config.CROSS_ENCODER_CALIBRATION_LOW/HIGH`), against the same pipeline
with rerank off.


In [1]:
import _setup as s
import pandas as pd

W1123 15:00:04.550000 25550 .venv/lib/python3.13/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [2]:
rerank_results = {}
for rerank in [False, True]:
    rerank_results[rerank] = s.run_eval(
        model="bge-small", chunking="section", mode="hybrid", rerank=rerank, warm_runs=5
    )

rerank_df = pd.DataFrame(
    {
        ("rerank" if r else "no rerank"): {
            **res["retrieval_accuracy"]["macro_avg"],
            "p50_total_ms": res["latency"]["query"]["macro_avg"]["total"]["p50_ms"],
        }
        for r, res in rerank_results.items()
    }
).T
rerank_df

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

,precision_at_5,precision_at_10,recall_at_10,mrr,ndcg_at_10,hit_at_1,p50_total_ms
no rerank,0.966667,0.766667,0.694444,1.0,0.917575,1.0,60.960181
rerank,0.966667,0.750000,0.680556,1.0,0.905143,1.0,248.983083


In [3]:
for name, row in rerank_df.iterrows():
    print(f"{name:<10} nDCG@10={row['ndcg_at_10']:.3f}  p50={row['p50_total_ms']:.1f}ms")

no rerank  nDCG@10=0.918  p50=61.0ms
rerank     nDCG@10=0.905  p50=249.0ms


**Finding:** with R19's cross-encoder now scoring the whole filtered pool instead of a top-20 subset, on fixed calibration bounds rather than a sigmoid, rerank still gives a small nDCG@10 *loss* here (0.918 -> 0.905) at roughly 4x the query latency (61ms -> 249ms p50) — the same direction as the pre-Step-11 measurement, though both numbers moved (the no-rerank baseline itself dropped from 0.944 to 0.918; see notebook 03's finding and `PLAN.md`'s Step 11 TODOs). `--rerank` stays off by default (see `PLAN.md`'s design-decisions table). Note also that `config.CROSS_ENCODER_CALIBRATION_LOW/HIGH` are still placeholder bounds, not yet tuned against this cross-encoder's actual logit distribution on this dataset the way the embedding models' bounds were tuned in Step 6 — tracked as a TODO in `PLAN.md`.
